In [1]:
!pip install -q google-generativeai

In [2]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [12]:
from google.colab import userdata
from google import genai

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

In [13]:
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

Learns patterns from data to make decisions.


In [6]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1elVAMyYyb4iURUAshKEr7WbC1BGDIUWWSz05MMsIG-M/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

In [7]:
# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"已記錄：日期: {today}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [8]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [9]:
new_grades = get_user_grades()

--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：1
請輸入 1 的成績：12
已記錄：日期: 2026-04-16, 科目: 1, 成績: 12

請輸入科目（或輸入 'q' 停止）：2
請輸入 2 的成績：53
已記錄：日期: 2026-04-16, 科目: 2, 成績: 53

請輸入科目（或輸入 'q' 停止）：3
請輸入 3 的成績：62
已記錄：日期: 2026-04-16, 科目: 3, 成績: 62

請輸入科目（或輸入 'q' 停止）：q


In [10]:
new_grades

[['2026-04-16', '1', 12], ['2026-04-16', '2', 53], ['2026-04-16', '3', 62]]

In [14]:
get_ai_summary(new_grades)


--- 正在呼叫 AI 模型生成摘要... ---


'好的，這是一份根據您提供的成績列表所做的簡單摘要與常見迷思整理，全程不對成績本身進行評分或判斷：\n\n---\n\n### 學生成績摘要與常見迷思整理\n\n**日期：** 2026-04-16\n\n---\n\n#### 1. 簡單成績摘要 (不評分，只做總結)\n\n本次成績列表包含2026年4月16日的三個科目表現：\n\n*   **科目1：** 12分\n*   **科目2：** 53分\n*   **科目3：** 62分\n\n**總結觀察：**\n這三項成績的數值分佈從12分到62分。其中一個科目獲得了相對較低的分數（12分），而另外兩個科目則取得了中等範圍的成績（53分和62分）。整體來看，學生在不同科目間的表現存在差異性。\n\n---\n\n#### 2. 成績解讀常見迷思整理 (不評分，只做總結)\n\n在檢視學生成績時，人們常會有一些預設的觀念，以下整理幾項常見的迷思及其澄清：\n\n1.  **迷思一：成績高低等同於學生的全部能力或努力程度。**\n    *   **澄清：** 成績是特定時間點、針對特定評量內容的表現回饋。它受到多種因素影響，例如考試當天的狀態、對該科目的興趣、學習策略、試題難易度、甚至身體狀況等，不能全面代表學生的潛力、學習過程中的真實努力或非學術領域的才華。\n\n2.  **迷思二：單一低分代表學生完全不理解或該科目沒有希望。**\n    *   **澄清：** 特定科目成績不理想，可能是因為該部分內容較為艱深、一時疏忽、考試焦慮、對題型不熟悉，或學習方法尚需調整。這不代表學生在該科目上完全沒有進步的潛力，也不是其整體學習狀況的定論。\n\n3.  **迷思三：成績是衡量學習成果的唯一標準。**\n    *   **澄清：** 學習是一個多元的過程，除了紙筆測驗的成績外，還包括了思考能力的培養、解決問題的能力、溝通協作、批判性思維、對知識的好奇心以及實踐應用等。這些能力往往無法完全透過單純的成績數字來量化。\n\n4.  **迷思四：成績一旦確定就無法改變，是永久性的標籤。**\n    *   **澄清：** 學習是一個持續且動態的過程。目前的成績僅代表學生在該時間點的學習狀態，隨著時間的推移、學習策略的調整、額外的努力或更有效的指導，學生的表現是完全有可能提升和改變的。成績更應被視為一種診斷工具，而非

In [17]:
def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)





        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(new_grades)
        print("\n--- 成績已成功寫入 Google Sheet。---")

        # 4. 獲取 AI 摘要並寫入 Google Sheet
        summary = get_ai_summary(new_grades)

        # 尋找第一行空白列
        next_row = len(ws.col_values(1)) + 1

        # 使用 update_cell() 方法逐一更新儲存格
        ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            ws.update_cell(next_row + i + 1, 1, line)

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：1
請輸入 1 的成績：15
已記錄：日期: 2026-04-16, 科目: 1, 成績: 15

請輸入科目（或輸入 'q' 停止）：2
請輸入 2 的成績：26
已記錄：日期: 2026-04-16, 科目: 2, 成績: 26

請輸入科目（或輸入 'q' 停止）：3
請輸入 3 的成績：60
已記錄：日期: 2026-04-16, 科目: 3, 成績: 60

請輸入科目（或輸入 'q' 停止）：1
請輸入 1 的成績：25
已記錄：日期: 2026-04-16, 科目: 1, 成績: 25

請輸入科目（或輸入 'q' 停止）：q

--- 成績已成功寫入 Google Sheet。---

--- 正在呼叫 AI 模型生成摘要... ---

--- AI 摘要已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
好的，這是一份根據您提供的成績列表所做的簡單摘要與常見迷思整理，其中不包含任何評分或判斷。

---

### 簡單摘要

這些成績紀錄發生在**2026年4月16日**，共包含**四筆資料**。

*   **科目涵蓋：** 涉及科目1（有兩筆紀錄）、科目2和科目3。
*   **分數分佈：** 四筆成績的分數範圍從**15分到60分**。
*   **平均分數：** 這四筆成績的算術平均值約為 **31.5分**。

**總結來說：** 這是一組來自同一日期的學習表現紀錄，涵蓋了不同科目，分數顯示出一定的分佈範圍。

---

### 常見迷思整理（關於成績）

在檢視任何成績時，人們往往會有一些常見的誤解，以下是幾點整理：

1.  **迷思：單一分數能全面評估學生能力。**
    *   **事實：** 成績只是在特定時間點、針對特定學習內容的一次表現紀錄。它無法反映學生的全部潛力、學習過程中的努力、克服困難的毅力，或在其他領域的才能。一個分數往往只是冰山一角。

2.  **迷思：分數高低直接等同於學生的智力或努力程度。**
    *   **事實：** 學習成果受到多